# Finetune BEATs (pretrain AS2M) trên mixture-36l

Nạp checkpoint BEATs pretrain `BEATs_iter3_plus_AS2M.pt` (self-supervised trên AudioSet 2M), gắn đầu phân loại 36 lớp, rồi **finetune toàn bộ mạng** trên bộ `mixture-36l` của dự án.

- Chỉ dùng BEATs: không có front-end FRCRN. Đầu vào là waveform mixture 4 giây. `BEATTrainer` nhận `separator=None`.
- Chọn epoch và early stop theo `macro_f1` trên `validation` đầy đủ. Báo cáo thêm `validation_clean` (đã bỏ các ytid AudioSet mà pretrain có thể đã thấy).
- Tập `test` không dùng trong notebook này.
- `RUN_TRAINING = False` (mặc định) chạy smoke test: 1 epoch, 64 mẫu, ghi vào `output/test/`. Đặt `True` để chạy thật, ghi vào `output/`.
- Chạy tiếp một lần dở dang: đặt `RESUME_RUN_ID` là tên thư mục lần chạy (ví dụ `beats-ft-2026-10-27-10-00-00`).

| Bước | Nội dung |
| --- | --- |
| 1 | Tên pipeline, chế độ chạy, `RunPaths`, logger |
| 2 | Cấu hình (`config/beat.yaml`), seed |
| 3 | Dataset `mixture-36l`: train, validation, validation_clean |
| 4 | Nạp BEATs pretrain |
| 5 | Dựng model, kiểm tra số lớp |
| 6 | Finetune, đánh giá, ghi kết quả |
| 7 | Bảng, biểu đồ, ma trận nhầm lẫn |

In [ ]:
# Chạy từ thư mục gốc của repo để import được `scripts` và `src`
import os
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "rules.MD").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
# (1) Tên pipeline, chế độ chạy, đường dẫn đầu ra (RunPaths), logger
import torch

from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

PIPELINE_NAME = "beats-ft"
RUN_TRAINING = False  # True: chạy thật; False: smoke test nhỏ ghi vào output/test
RESUME_RUN_ID = None  # ví dụ "beats-ft-2026-10-27-10-00-00" để chạy tiếp lần chạy cũ

output_dir = Path("output") if RUN_TRAINING else Path("output") / "test"
if RESUME_RUN_ID:
    run = RunPaths.from_run_id(RESUME_RUN_ID, output_dir)
else:
    run = RunPaths.new(PIPELINE_NAME if RUN_TRAINING else f"{PIPELINE_NAME}-smoke", output_dir)
run.create()

logger = setup_logger(PIPELINE_NAME, run.log_file)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("run %s | device %s | RUN_TRAINING=%s", run.run_id, device, RUN_TRAINING)

In [ ]:
# (2) Cấu hình finetune (mục train của config/beat.yaml) và seed; smoke test thu nhỏ epoch, batch và số mẫu
from dataclasses import replace

import yaml

from scripts.finetune import BEAT_CONFIG, CLIP_SECONDS, SMOKE_BATCH, SMOKE_SAMPLES
from src.train import BEATTrainConfig
from src.utils.seed import set_seed

config = BEATTrainConfig.from_yaml(BEAT_CONFIG)
SEED = yaml.safe_load(BEAT_CONFIG.read_text(encoding="utf-8"))["train"]["seed"]
NUM_WORKERS = 4
MAX_SAMPLES = None
if not RUN_TRAINING:
    config = replace(config, epochs=1, batch_size=SMOKE_BATCH, eval_batch_size=SMOKE_BATCH)
    MAX_SAMPLES, NUM_WORKERS = SMOKE_SAMPLES, 0
set_seed(SEED)
print(config)

## (3) Dataset mixture-36l

`mixture-36l` có 36 lớp nhiễu, mỗi clip là hỗn hợp giọng nói và nhiễu ở một mức SNR. Clip được cắt hoặc đệm về 4 giây.
`validation_clean` là `validation` sau khi bỏ các sample có `noise_ytid` nằm trong danh sách AudioSet của pretrain (`validation_clean.csv`).
Tập `test` không được nạp.

In [ ]:
# (3) Nạp mixture-36l (tự tải từ Kaggle khi chưa có) và ba loader: train, validation, validation_clean
from scripts.download_dataset import ensure_dataset
from scripts.dynamic_mix_36l import mixture_class_names
from scripts.load_dataset import MIXTURE_KEY, Mixture36Dataset, build_dataloader, subsample

dataset_root = ensure_dataset(MIXTURE_KEY, Path("datasets"))
class_names = mixture_class_names(dataset_root)


def build_loader(split: str, batch_size: int, shuffle: bool, sample_ids_file: Path | None = None):
    dataset = Mixture36Dataset(
        dataset_root, split=split, duration=CLIP_SECONDS, load_noise=False, sample_ids_file=sample_ids_file
    )
    return build_dataloader(subsample(dataset, MAX_SAMPLES), batch_size, shuffle, NUM_WORKERS)


loaders = {
    "train": build_loader("train", config.batch_size, True),
    "validation": build_loader("validation", config.eval_batch_size, False),
    "validation_clean": build_loader(
        "validation", config.eval_batch_size, False, dataset_root / "validation_clean.csv"
    ),
}
print(f"mixture-36l: {dataset_root} | {len(class_names)} lớp")
for name, loader in loaders.items():
    print(f"{name}: {len(loader.dataset)} clip")

## (4) Nạp BEATs pretrain

Checkpoint `BEATs_iter3_plus_AS2M.pt` (khoá `beats-pretrain` trong `config/url.yaml`) là bản self-supervised, chưa finetune trên nhãn. Lớp `head` (đầu phân loại 36 lớp) được khởi tạo ngẫu nhiên; `load_pretrained` chỉ cho phép thiếu các khoá `head.*`.

In [ ]:
# (4) Nạp trọng số pretrain vào BEATClassifier (cấu hình từ config/beat.yaml)
from scripts.load_pretrain import download_pretrained
from src.model.beat import BEATClassifier

pretrain_path = download_pretrained("beats")
beats = BEATClassifier.from_yaml(BEAT_CONFIG)
pretrain_report = beats.load_pretrained(pretrain_path)
print(f"checkpoint: {pretrain_path}")
print(f"missing {len(pretrain_report.missing)} | unexpected {len(pretrain_report.unexpected)}")

## (5) Dựng model

Mở toàn bộ tham số để finetune (không đóng băng backbone). Kiểm tra số lớp của model khớp với dataset trước khi huấn luyện.

In [ ]:
# (5) Đưa model lên device, mở tham số để finetune, kiểm tra số lớp
from scripts.finetune import set_trainable

if beats.config.num_classes != len(class_names):
    raise ValueError(f"model có {beats.config.num_classes} lớp, dataset có {len(class_names)} lớp")
beats = set_trainable(beats.to(device), True)
print(
    f"{beats.config.encoder_layers} layer | {beats.num_parameters(trainable_only=True) / 1e6:.2f}M tham số huấn luyện"
)

## (6) Finetune, đánh giá và ghi kết quả

Mỗi epoch ghi `best.pt` (khi `macro_f1` trên validation tăng) và `resume.pt` (để chạy tiếp). Sau khi dừng, model được khôi phục về epoch tốt nhất rồi đánh giá trên `validation` và `validation_clean`.
Nếu lần chạy đã có `result.csv`, ô này dừng với lỗi để không ghi đè lịch sử.

In [ ]:
# (6) Finetune BEATs, đánh giá validation và validation_clean, ghi history và result.csv
import json
import time

from scripts.finetune import METRIC_NAMES
from scripts.run_baselines import write_result_csv
from src.train import BEATTrainer

if run.result_file.is_file():
    raise FileExistsError(f"{run.result_file} đã có; đặt RESUME_RUN_ID để tiếp tục hoặc tạo lần chạy mới")

checkpoint_path = run.checkpoint_path("beats")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
resume_path = checkpoint_path.with_name("resume.pt")
trainer = BEATTrainer(beats, config, device, logger, "beats", checkpoint_path, resume_path=resume_path)
if resume_path.is_file():
    trainer.load_resume(resume_path)

started = time.time()
fit = trainer.fit(loaders["train"], loaders["validation"])
trainer._in_fit = False  # đánh giá cuối không được ghi thêm vào history (cùng cách với scripts/finetune.py)
scores = {name: trainer.evaluate(loader, desc=f"beats - {name}") for name, loader in loaders.items() if name != "train"}
confusion = trainer.confusion_matrix(loaders["validation"], desc="beats - confusion")

payload = {
    **fit.to_dict(),
    "val": scores,
    "confusion_matrix": confusion.tolist(),
    "train_seconds": round(time.time() - started, 1),
}
run.history_path("beats").write_text(json.dumps(payload, indent=2), encoding="utf-8")
rows = [{"split": name, **values} for name, values in scores.items()]
write_result_csv(rows, run.result_file, ("split", *METRIC_NAMES))
logger.info(
    "beats: best %s %.4f @ epoch %d / %d",
    fit.selection_metric,
    fit.best_score,
    fit.best_epoch,
    fit.epochs_run,
)
logger.info("beats: kết quả ghi tại %s", run.result_file)

## (7) Bảng kết quả và biểu đồ

Bảng có hai dòng: `validation` (dùng để chọn epoch) và `validation_clean` (báo cáo). Chênh lệch giữa hai dòng cho biết mức ảnh hưởng của ytid rò rỉ.

In [ ]:
# (7) Bảng kết quả, đường cong loss và macro_F1 theo epoch, ma trận nhầm lẫn của validation
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from scripts.run_baselines import markdown_table
from src.utils.plots import plot_confusion_matrix

display(Markdown(markdown_table(rows, ("split", *METRIC_NAMES))))

epochs = [entry["epoch"] for entry in fit.history]
figure, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(10, 3.6))
ax_loss.plot(epochs, [entry["train_loss"] for entry in fit.history], marker="o", label="train loss")
ax_f1.plot(epochs, [entry["val_macro_f1"] for entry in fit.history], marker="o", label="validation macro_f1")
for ax in (ax_loss, ax_f1):
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend()
figure.tight_layout()
figure.savefig(run.figure_dir / "beats_curves.png", dpi=120)
plt.show()

confusion_path = plot_confusion_matrix(
    confusion, class_names, run.figure_dir / "beats_confusion.png", title="beats validation"
)
display(Image(filename=str(confusion_path)))
print(f"lần chạy: {run.run_id} | kết quả: {run.result_file}")